In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Union, Optional, Dict
import pyranges as pr
# REQUIRED: choose context & genotype
context = "CG"     # e.g., "CG", "CHG", "CHH"
mutant  = "col"    # e.g., "met", "rdd", etc.
# Python 3.8-compatible type alias
PathLike = Union[str, Path]

In [2]:
def normalize_chr(x):
    """Strip leading 'chr'/'Chr' if present; return string."""
    if pd.isna(x):
        return x
    s = str(x).strip()
    if s.lower().startswith("chr"):
        s = s[3:]
    return s

# Regexes to pull IDs from GFF attrs
_GENE_ID_RE = re.compile(r'gene_id\s+"([^"]+)"')
_TRANSCRIPT_ID_RE = re.compile(r'transcript_id\s+"([^"]+)"')
_ID_RE = re.compile(r'ID=([^;]+)')

def parse_feature_id(attr_str: str) -> str:
    """Try gene_id, then transcript_id, then generic ID=; else empty."""
    if pd.isna(attr_str):
        return ""
    m = _GENE_ID_RE.search(attr_str)
    if m: return m.group(1)
    m = _TRANSCRIPT_ID_RE.search(attr_str)
    if m: return m.group(1)
    m = _ID_RE.search(attr_str)
    return m.group(1) if m else ""

def load_feature_gff(path: PathLike,
                     feature_filter: Optional[set] = None,
                     id_parser=parse_feature_id) -> pd.DataFrame:
    """
    Robust GFF loader (works w/ headered or headerless files).
    - Skips comment lines starting with '#'.
    - If header row present & contains 'chr' or 'seqid', we honor it.
    - Otherwise we assign standard 9-column GFF names.

    Returns DataFrame with canonical columns: chr,start,end,feature_id
    """
    # Try reading w/ header=None first to be safe
    raw = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        header=None,
        dtype=str,
        na_filter=False,
        engine="python"
    )

    # If the first row looks like a header (contains "chr" or "seqid" etc.), re-read with header=0
    first_vals = [v.lower() for v in raw.iloc[0].tolist()]
    header_like = any(x in first_vals for x in ("chr", "seqid", "source", "feature", "type"))
    if header_like:
        df = pd.read_csv(
            path,
            sep="\t",
            comment="#",
            header=0,
            dtype=str,
            na_filter=False,
            engine="python"
        )
        df.columns = [c.lower() for c in df.columns]
    else:
        # assign standard GFF colnames
        std_cols = ["seqid","source","type","start","end","score","strand","phase","attrs"]
        # raw may have >9 columns if attributes contain extra tabs; truncate
        if raw.shape[1] < 5:
            raise ValueError(f"{path}: fewer than 5 columns found; not a valid GFF?")
        # Trim or pad to 9
        ncols = min(len(std_cols), raw.shape[1])
        df = raw.iloc[:, :ncols].copy()
        df.columns = std_cols[:ncols]
        df.columns = [c.lower() for c in df.columns]

    # Map seqid/type synonyms
    cols = df.columns.tolist()
    colmap = {}
    if "chr" in cols:
        colmap["chr"] = "chr"
    elif "seqid" in cols:
        colmap["seqid"] = "chr"
    elif "chrom" in cols:
        colmap["chrom"] = "chr"
    else:
        raise ValueError(f"{path}: cannot find chromosome column (looked for chr/seqid/chrom).")

    # unify start/end
    if "start" not in cols or "end" not in cols:
        raise ValueError(f"{path}: missing start/end after parsing; got columns {cols}")

    df = df.rename(columns=colmap)

    # Feature column name: 'feature' (if present) else 'type' else None
    if "feature" not in df.columns and "type" in df.columns:
        df = df.rename(columns={"type": "feature"})

    # Normalize types & optional filter
    if "feature" in df.columns and feature_filter:
        df = df[df["feature"].isin(feature_filter)].copy()

    # Convert types
    df["chr"] = df["chr"].map(normalize_chr)
    df["start"] = df["start"].astype(int)
    df["end"]   = df["end"].astype(int)

    # Feature ID from attrs (if present)
    if "attrs" in df.columns:
        df["feature_id"] = df["attrs"].map(id_parser)
    else:
        df["feature_id"] = [f"feat_{i}" for i in range(len(df))]

    # QC
    bad = df["end"] < df["start"]
    if bad.any():
        raise ValueError(f"{path}: {bad.sum()} rows with end < start.")

    df = df.sort_values(["chr","start","end"], kind="mergesort").reset_index(drop=True)
    return df[["chr","start","end","feature_id"]]

def boolean_overlap(win_df: pd.DataFrame, feat_df: pd.DataFrame) -> pd.Series:
    """
    Return bool Series aligned to win_df index: does window overlap ANY feature?
    Coordinates are 1-based inclusive in both tables.
    """
    flag = pd.Series(False, index=win_df.index)

    # Pre-split windows by chromosome
    grp = win_df.groupby("chr", sort=False)
    chrom_to_rows = {c: ix.values for c, ix in grp.groups.items()}
    chrom_to_starts = {c: win_df.loc[ix, "start"].to_numpy() for c, ix in grp.groups.items()}
    chrom_to_ends   = {c: win_df.loc[ix, "end"].to_numpy()   for c, ix in grp.groups.items()}

    # Iterate feature chromosomes
    for chrom, fsub in feat_df.groupby("chr", sort=False):
        if chrom not in chrom_to_rows:
            continue
        win_idx = chrom_to_rows[chrom]
        starts  = chrom_to_starts[chrom]
        ends    = chrom_to_ends[chrom]

        fstarts = fsub["start"].to_numpy()
        fends   = fsub["end"].to_numpy()

        # Binary search window slices per feature interval
        i0 = np.searchsorted(ends, fstarts, side="left")
        i1 = np.searchsorted(starts, fends, side="right") - 1

        for fs, fe, s0, s1 in zip(fstarts, fends, i0, i1):
            if s0 >= len(starts) or s1 < 0 or s0 > s1:
                continue
            cand_starts = starts[s0:s1+1]
            cand_ends   = ends[s0:s1+1]
            mask = (cand_starts <= fe) & (cand_ends >= fs)
            if not mask.any():
                continue
            abs_rows = win_idx[s0 + np.nonzero(mask)[0]]
            flag.iloc[abs_rows] = True

    return flag



In [3]:
%pwd

'/ceph/MethDev/JW240627--at-snmCT_with_TE/mCT_with_TE/kay'

In [4]:
df = pd.read_pickle("./data/gffs_sorted.pkl")
df

,cluster,chr,start,end,score,c,t,n
0,0,1,101,200,0.8951,350,41,6
1,0,1,301,400,0.5487,62,51,2
2,0,1,401,500,0.8246,47,10,1
3,0,1,501,600,0.7206,98,38,3
4,0,1,601,700,0.8982,203,23,6
...,...,...,...,...,...,...,...,...
16297919,9,5,26974801,26974900,0.8000,32,8,10
16297920,9,5,26974901,26975000,1.0000,4,0,2
16297921,9,5,26975101,26975200,1.0000,4,0,2
16297922,9,5,26975201,26975300,0.9773,43,1,14


In [5]:
df.reset_index(drop=True, inplace=True)
df

,cluster,chr,start,end,score,c,t,n
0,0,1,101,200,0.8951,350,41,6
1,0,1,301,400,0.5487,62,51,2
2,0,1,401,500,0.8246,47,10,1
3,0,1,501,600,0.7206,98,38,3
4,0,1,601,700,0.8982,203,23,6
...,...,...,...,...,...,...,...,...
16258583,9,5,26974801,26974900,0.8000,32,8,10
16258584,9,5,26974901,26975000,1.0000,4,0,2
16258585,9,5,26975101,26975200,1.0000,4,0,2
16258586,9,5,26975201,26975300,0.9773,43,1,14


In [6]:
# 1. Show memory per column (including object-dtypes like strings)
mem_per_col = df.memory_usage(index=True, deep=True)
print(mem_per_col)

# 2. Sum it up for the total footprint
total = mem_per_col.sum()
print(f"\nTotal memory usage: {total/1024**2:.2f} MB")

Index           128
cluster    16258588
chr        16258588
start      65034352
end        65034352
score      65034352
c          65034352
t          65034352
n          16258588
dtype: int64

Total memory usage: 356.62 MB


In [7]:
#load feature gffs

In [8]:
feature_gffs = {
    "euc_gene": "../master_gffs/andy_lowercase/euc_genes.gff",
    "het_gene": "../master_gffs/andy_lowercase/het_genes.gff",
    "euc_TE": "../master_gffs/andy_lowercase/euc_TEs.gff",
    "het_TE": "../master_gffs/andy_lowercase/het_TEs.gff",
}

# Load present feature sets
# We will produce a dict key -> DataFrame (or None if missing)
feat_dfs = {}
for key, path in feature_gffs.items():
    feat_dfs[key.lower()] = load_feature_gff(path)
    print(f"Loaded {key}: {len(feat_dfs[key.lower()]):,} intervals")


Loaded euc_gene: 29,560 intervals
Loaded het_gene: 3,504 intervals
Loaded euc_TE: 14,789 intervals
Loaded het_TE: 16,400 intervals


In [9]:
feat_dfs

{'euc_gene':       chr     start       end feature_id
 0       1      3631      5899  AT1G01010
 1       1      6788      9130  AT1G01020
 2       1     11101     11372  AT1G03987
 3       1     11649     13714  AT1G01030
 4       1     23121     31227  AT1G01040
 ...    ..       ...       ...        ...
 29555   5  26966885  26967079  AT5G09945
 29556   5  26967378  26969400  AT5G67630
 29557   5  26969516  26970668  AT5G67640
 29558   5  26971389  26971689  AT5G09955
 29559   5  26972177  26972644  AT5G09965
 
 [29560 rows x 4 columns],
 'het_gene':      chr     start       end feature_id
 0      1  12398428  12401036  AT1G34065
 1      1  12402283  12403209  AT1G34070
 2      1  12411670  12412514  AT1G34095
 3      1  12417012  12421792  AT1G34110
 4      1  12426700  12429714  AT1G34120
 ...   ..       ...       ...        ...
 3499   5  15282928  15283377  AT5G05665
 3500   5  15283692  15285837  AT5G38260
 3501   5  15287882  15288207  AT5G05685
 3502   5  15289032  15290252  AT

In [10]:
import pandas as pd
import numpy as np
import re

def _chr_to_int(s):
    s = s.astype(str).str.replace(r'^chr', '', regex=True)
    return pd.to_numeric(s, errors='coerce').astype('Int64')

def boolean_overlap(win_df: pd.DataFrame, feat_df: pd.DataFrame) -> pd.Series:
    # normalize chr to integer-like on both sides (handles 'chr1' and '1')
    win = win_df.copy()
    feat = feat_df.copy()
    win['chr']  = _chr_to_int(win['chr'])
    feat['chr'] = _chr_to_int(feat['chr'])

    # sort windows per chromosome so searchsorted works
    win = win.sort_values(['chr', 'start', 'end'])
    flag = pd.Series(False, index=win.index)

    # build per-chrom caches in one pass (keeps arrays aligned)
    chrom_cache = {}
    for chrom, sub in win.groupby('chr', sort=False):
        chrom_cache[chrom] = {
            'rows':   sub.index.to_numpy(),
            'starts': sub['start'].to_numpy(),
            'ends':   sub['end'].to_numpy()
        }

    # iterate features per chromosome
    for chrom, fsub in feat.groupby('chr', sort=False):
        if chrom not in chrom_cache or fsub.empty:
            continue
        rows   = chrom_cache[chrom]['rows']
        starts = chrom_cache[chrom]['starts']
        ends   = chrom_cache[chrom]['ends']

        fstarts = fsub['start'].to_numpy()
        fends   = fsub['end'].to_numpy()

        # candidate window index range for each feature
        i0 = np.searchsorted(ends,   fstarts, side='left')
        i1 = np.searchsorted(starts, fends,   side='right') - 1

        for fs, fe, lo, hi in zip(fstarts, fends, i0, i1):
            if lo > hi or lo >= len(starts) or hi < 0:
                continue
            cs = starts[lo:hi+1]
            ce = ends[lo:hi+1]
            mask = (cs <= fe) & (ce >= fs)  # 1-based inclusive overlap
            if mask.any():
                hit_rows = rows[lo + np.nonzero(mask)[0]]
                flag.loc[hit_rows] = True   # <-- use loc (labels), not iloc

    # return aligned to the original win_df (in its current order)
    return flag.reindex(win_df.index, fill_value=False)


In [49]:
import pandas as pd
import numpy as np

def boolean_overlap(win_df: pd.DataFrame, feat_df: pd.DataFrame) -> pd.Series:
    """
    Return a Boolean Series aligned to win_df.index: True if that window
    overlaps ANY feature in feat_df. Coordinates are 1-based inclusive.
    """
    # 0) Ensure chr is the same dtype in both
    win_chr  = win_df["chr"]
    feat_chr = feat_df["chr"]
    if win_chr.dtype != feat_chr.dtype:
        # coerce both to integer, dropping non-digits as NaN
        win_df  = win_df.copy()
        feat_df = feat_df.copy()
        win_df["chr"]  = pd.to_numeric(win_df["chr"], errors="coerce").astype("Int64")
        feat_df["chr"] = pd.to_numeric(feat_df["chr"], errors="coerce").astype("Int64")

    # 1) initialize all False
    flag = pd.Series(False, index=win_df.index)

    # 2) group windows by chromosome
    grp = win_df.groupby("chr", sort=False)
    chrom_to_rows   = {c: idx.values for c, idx in grp.groups.items()}
    chrom_to_starts = {c: win_df.loc[idx, "start"].to_numpy() for c, idx in grp.groups.items()}
    chrom_to_ends   = {c: win_df.loc[idx, "end"].to_numpy()   for c, idx in grp.groups.items()}

    # 3) for each feature-chromosome
    for chrom, fsub in feat_df.groupby("chr", sort=False):
        if chrom not in chrom_to_rows:
            # no windows on this chrom
            continue

        starts = chrom_to_starts[chrom]
        ends   = chrom_to_ends[chrom]
        win_idx = chrom_to_rows[chrom]

        fstarts = fsub["start"].to_numpy()
        fends   = fsub["end"].to_numpy()

        # 4) for each feature, find candidate windows via binary search
        i0 = np.searchsorted(ends,   fstarts, side="left")
        i1 = np.searchsorted(starts, fends,   side="right") - 1

        for fs, fe, lo, hi in zip(fstarts, fends, i0, i1):
            if lo > hi or lo >= len(starts) or hi < 0:
                continue
            # slice candidate windows
            cs = starts[lo:hi+1]
            ce = ends  [lo:hi+1]
            # check for real overlap
            mask = (cs <= fe) & (ce >= fs)
            if not mask.any():
                continue
            # mark those in the global flag
            hit_rows = win_idx[lo + np.nonzero(mask)[0]]
            flag.iloc[hit_rows] = True

    return flag


In [11]:
win_unique = df

In [12]:
# Initialize all-False series
flag_euc_gene = pd.Series(False, index=win_unique.index)
flag_het_gene = pd.Series(False, index=win_unique.index)
flag_euc_TE   = pd.Series(False, index=win_unique.index)
flag_het_TE   = pd.Series(False, index=win_unique.index)

# Fill available
flag_euc_gene = boolean_overlap(win_unique, feat_dfs["euc_gene"])
flag_het_gene = boolean_overlap(win_unique, feat_dfs["het_gene"])
flag_euc_TE = boolean_overlap(win_unique, feat_dfs["euc_te"])
flag_het_TE = boolean_overlap(win_unique, feat_dfs["het_te"])

# Combine
win_unique_flags = win_unique.assign(
    flag_euc_gene = flag_euc_gene.values,
    flag_het_gene = flag_het_gene.values,
    flag_euc_TE   = flag_euc_TE.values,
    flag_het_TE   = flag_het_TE.values,
)
win_unique_flags


,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE
0,0,1,101,200,0.8951,350,41,6,False,False,False,False
1,0,1,301,400,0.5487,62,51,2,False,False,False,False
2,0,1,401,500,0.8246,47,10,1,False,False,False,False
3,0,1,501,600,0.7206,98,38,3,False,False,False,False
4,0,1,601,700,0.8982,203,23,6,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
16258583,9,5,26974801,26974900,0.8000,32,8,10,False,False,False,False
16258584,9,5,26974901,26975000,1.0000,4,0,2,False,False,False,False
16258585,9,5,26975101,26975200,1.0000,4,0,2,False,False,False,False
16258586,9,5,26975201,26975300,0.9773,43,1,14,False,False,False,False


In [51]:
flag_euc_gene.sum()

531161

In [24]:
win_unique_flags.loc[97]

cluster               0
chr                   1
start             11901
end               12000
score            0.0232
c                     6
t                   253
n                     6
flag_euc_gene      True
flag_het_gene     False
flag_euc_TE        True
flag_het_TE       False
Name: 97, dtype: object

In [21]:
win_unique_flags[win_unique_flags['flag_euc_TE']==True]

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE
97,0,1,11901,12000,0.0232,6,253,6,True,False,True,False
140,0,1,16801,16900,0.0214,10,458,10,False,False,True,False
141,0,1,16901,17000,0.0097,2,205,6,False,False,True,False
142,0,1,17101,17200,0.0000,0,1,1,False,False,True,False
143,0,1,17301,17400,0.0000,0,5,2,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
16258506,9,5,26966001,26966100,0.0000,0,3,2,True,False,True,False
16258507,9,5,26966101,26966200,0.0000,0,5,2,False,False,True,False
16258508,9,5,26966401,26966500,0.0000,0,11,4,False,False,True,False
16258509,9,5,26966501,26966600,0.0000,0,6,2,False,False,True,False


In [22]:
win_unique_flags[(win_unique_flags['flag_euc_TE']==True) & (win_unique_flags['cluster']==16)]

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE
7615218,16,1,11901,12000,0.0000,0,12,6,True,False,True,False
7615258,16,1,16801,16900,0.0000,0,12,10,False,False,True,False
7615259,16,1,16901,17000,0.0000,0,4,4,False,False,True,False
7615260,16,1,17501,17600,0.3333,1,2,3,False,False,True,False
7615261,16,1,17601,17700,0.0000,0,1,1,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8536155,16,5,26966001,26966100,0.0000,0,3,2,True,False,True,False
8536156,16,5,26966101,26966200,0.0000,0,2,2,False,False,True,False
8536157,16,5,26966401,26966500,0.0000,0,5,4,False,False,True,False
8536158,16,5,26966501,26966600,0.0000,0,5,2,False,False,True,False


In [25]:
out_path = f"./data/annotated_full_{mutant}.{context}_2.fast.tsv"
win_unique_flags.to_csv(out_path, sep="\t", index=False)
print(f"Wrote: {out_path}")


Wrote: ./data/annotated_full_col.CG_2.fast.tsv
